In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Wazirpur_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,NaN,191.0,213.0,116.0,94.0,129.0,138.0,130.0,148.0,176.0,408.0,319.0
1,2,390.0,226.0,263.0,209.0,87.0,123.0,132.0,NaN,149.0,164.0,425.0,368.0
2,3,414.0,232.0,185.0,227.0,148.0,127.0,124.0,NaN,NaN,186.0,489.0,338.0
3,4,369.0,NaN,NaN,137.0,127.0,177.0,208.0,110.0,145.0,218.0,460.0,334.0
4,5,359.0,274.0,153.0,177.0,227.0,167.0,153.0,NaN,133.0,201.0,474.0,309.0
5,6,419.0,326.0,150.0,208.0,276.0,141.0,99.0,127.0,132.0,264.0,461.0,313.0
6,7,398.0,330.0,221.0,195.0,211.0,NaN,161.0,125.0,119.0,305.0,NaN,351.0
7,8,379.0,171.0,NaN,191.0,146.0,167.0,128.0,154.0,122.0,242.0,NaN,362.0
8,9,452.0,247.0,120.0,295.0,230.0,180.0,NaN,153.0,89.0,222.0,459.0,343.0
9,10,419.0,231.0,221.0,296.0,218.0,174.0,NaN,NaN,109.0,NaN,305.0,324.0


In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    35 non-null     float64
 2   February   32 non-null     float64
 3   March      33 non-null     float64
 4   April      34 non-null     float64
 5   May        36 non-null     float64
 6   June       31 non-null     float64
 7   July       25 non-null     float64
 8   August     29 non-null     float64
 9   September  33 non-null     float64
 10  October    35 non-null     float64
 11  November   32 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [5]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,295.428571,191.00,213.000000,116.0,94.0,129.0,138.00,130.000000,148.000000,176.0,408.0,319.0
1,2,390.000000,226.00,263.000000,209.0,87.0,123.0,132.00,140.448276,149.000000,164.0,425.0,368.0
2,3,414.000000,232.00,185.000000,227.0,148.0,127.0,124.00,140.448276,126.242424,186.0,489.0,338.0
3,4,369.000000,220.25,169.545455,137.0,127.0,177.0,104.16,110.000000,145.000000,218.0,460.0,334.0
4,5,359.000000,274.00,153.000000,177.0,227.0,167.0,153.00,140.448276,133.000000,201.0,474.0,309.0
